# 🦆 Entraînement de l'IA Sexe (male / femelle) (classification)

Ce notebook fait tourner l'entraînement sur un **GPU gratuit** fourni par Google Colab.

**Avant de commencer :**
1. En haut du notebook : **Exécution** → **Modifier le type d'exécution** → sélectionne **GPU** (T4 gratuit) → Enregistrer
2. Sur ton PC : compresse le dossier `dataset_sexe_migrated` en `dataset_sexe.zip` (le dossier doit contenir directement les sous-dossiers de classes, ex. `dataset_sexe_migrated/femelle/`, `dataset_sexe_migrated/male/`, ...)

⚠️ **Déséquilibre + classe manquante dans tes données actuelles** : 286 images `femelle` pour 105 `male` (ratio ~2,7:1) — gérable avec la pondération de classes ci-dessous. Plus important : la classe `indetermine_juvenile` prévue dans `ANNOTATION_GUIDE.md` n'a **aucun exemple** dans le dataset actuel. Ce notebook entraîne donc un classifieur à 2 classes (`male`/`femelle`) uniquement. Si tu annotes des juvéniles plus tard, il faudra réentraîner à 3 classes.


## 1. Installer les dépendances

In [ ]:
!pip install -q scikit-learn matplotlib

import torch
print("GPU disponible :", torch.cuda.is_available())
print("Device :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (va dans Exécution > Modifier le type d'exécution)")


## 2. Importer ton dataset

Cette cellule ouvre un bouton pour uploader ton fichier `dataset_sexe.zip`.

In [ ]:
from google.colab import files
import zipfile
import os

print("Sélectionne ton fichier dataset_sexe.zip...")
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]

DATA_DIR = "/content/dataset_sexe"
os.makedirs(DATA_DIR, exist_ok=True)
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(DATA_DIR)

print(f"\n✅ Dataset extrait dans {DATA_DIR}")
!find {DATA_DIR} -maxdepth 3


⚠️ **Vérifie la sortie ci-dessus** : tu dois voir directement les dossiers de classes (un par espèce/sexe/couleur, ex. `milouin/`, `morillon/`). Si tu vois un niveau de dossier en trop (ex. `dataset_xxx/dataset_xxx_migrated/milouin/...`), exécute la cellule suivante pour corriger automatiquement, sinon passe à la suite.

In [ ]:
# À exécuter UNIQUEMENT si tu as un dossier en double
import shutil

# Cherche un unique sous-dossier contenant lui-même les dossiers de classes
entries = [e for e in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, e))]
if len(entries) == 1:
    nested = os.path.join(DATA_DIR, entries[0])
    for item in os.listdir(nested):
        shutil.move(os.path.join(nested, item), DATA_DIR)
    shutil.rmtree(nested)
    print("✅ Dossier en double corrigé.")
else:
    print("Pas de dossier en double détecté, rien à faire.")

print("\nContenu final :", os.listdir(DATA_DIR))


## 3. Vérifier la répartition des classes

On compte les images par classe **avant** d'entraîner quoi que ce soit — **Déséquilibre + classe manquante dans tes données actuelles** : 286 images `femelle` pour 105 `male` (ratio ~2,7:1) — gérable avec la pondération de classes ci-dessous. Plus important : la classe `indetermine_juvenile` prévue dans `ANNOTATION_GUIDE.md` n'a **aucun exemple** dans le dataset actuel. Ce notebook entraîne donc un classifieur à 2 classes (`male`/`femelle`) uniquement. Si tu annotes des juvéniles plus tard, il faudra réentraîner à 3 classes.

In [ ]:
from collections import Counter
from pathlib import Path

class_dirs = sorted([d for d in Path(DATA_DIR).iterdir() if d.is_dir()])
counts = {d.name: len(list(d.glob("*"))) for d in class_dirs}

print("Répartition des classes :")
total = sum(counts.values())
for name, n in counts.items():
    pct = 100 * n / total if total else 0
    print(f"  {name:20s} : {n:5d} images ({pct:.1f}%)")

if len(counts) < 2:
    raise ValueError("❌ Il faut au moins 2 classes (2 sous-dossiers) pour entraîner un classifieur.")

ratio = max(counts.values()) / max(1, min(counts.values()))
if ratio > 5:
    print(f"\n⚠️  Déséquilibre important détecté (ratio {ratio:.1f}:1). "
          f"On va compenser avec une pondération de classes dans la loss, "
          f"mais le résultat sur la classe minoritaire restera moins fiable "
          f"qu'avec plus d'exemples.")


## 4. Préparer les données (split stratifié + augmentation)

On utilise un split **stratifié** (pas un `random_split` classique) pour garantir que chaque classe — y compris la minoritaire — soit bien représentée à la fois en train et en val, même avec peu d'exemples. On ajoute aussi de la data augmentation légère côté train (flip, rotation, jitter de couleur) pour aider avec le peu de données disponibles.

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split

SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
VAL_RATIO = 0.2

torch.manual_seed(SEED)
np.random.seed(SEED)

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Deux instances du même ImageFolder (transforms différents pour train/val)
base_dataset = datasets.ImageFolder(DATA_DIR)
CLASSES = base_dataset.classes
print("Classes (ordre utilisé par le modèle) :", CLASSES)

indices = np.arange(len(base_dataset))
labels = np.array(base_dataset.targets)

train_idx, val_idx = train_test_split(
    indices, test_size=VAL_RATIO, random_state=SEED, stratify=labels
)

train_dataset_full = datasets.ImageFolder(DATA_DIR, transform=train_transforms)
val_dataset_full = datasets.ImageFolder(DATA_DIR, transform=val_transforms)

train_dataset = Subset(train_dataset_full, train_idx)
val_dataset = Subset(val_dataset_full, val_idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nTrain : {len(train_dataset)} images | Val : {len(val_dataset)} images")

# Vérifie que chaque classe est bien représentée en val
val_labels = labels[val_idx]
for i, c in enumerate(CLASSES):
    n_val = (val_labels == i).sum()
    print(f"  {c:20s} : {n_val} en val")
    if n_val == 0:
        print(f"    ⚠️  Aucun exemple de '{c}' en validation, les métriques pour cette classe seront vides.")


## 5. Construire le modèle (ResNet18, transfer learning)

In [ ]:
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
model = model.to(device)

# Pondération de classes : compense le déséquilibre en donnant plus de poids
# à l'erreur sur les classes minoritaires dans la loss.
class_counts_train = np.bincount(labels[train_idx], minlength=len(CLASSES))
class_weights = torch.tensor(
    [len(train_idx) / (len(CLASSES) * max(1, c)) for c in class_counts_train],
    dtype=torch.float32,
).to(device)
print("Poids de classes (loss) :", dict(zip(CLASSES, class_weights.cpu().numpy().round(2))))

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)


## 6. Lancer l'entraînement

In [ ]:
EPOCHS = 25
PATIENCE = 7  # arrêt anticipé si la loss de validation ne s'améliore plus

history = {"train_loss": [], "val_loss": [], "val_acc": []}
best_val_loss = float("inf")
best_state = None
epochs_no_improve = 0

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    train_loss = running_loss / len(train_dataset)

    model.eval()
    val_loss = 0.0
    correct = 0
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            val_loss += loss.item() * inputs.size(0)
            correct += (outputs.argmax(1) == targets).sum().item()
    val_loss /= len(val_dataset)
    val_acc = correct / len(val_dataset)

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Époque {epoch+1:3d}/{EPOCHS} - train_loss: {train_loss:.4f} - val_loss: {val_loss:.4f} - val_acc: {val_acc:.3f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"\n⏹️  Arrêt anticipé : pas d'amélioration depuis {PATIENCE} époques.")
            break

# Recharge les meilleurs poids (meilleure val_loss), pas les derniers
model.load_state_dict(best_state)
print(f"\n✅ Entraînement terminé. Meilleure val_loss : {best_val_loss:.4f}")


## 7. Courbes d'entraînement

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Époque")
axes[0].legend()

axes[1].plot(history["val_acc"], label="val_acc", color="green")
axes[1].set_title("Précision (validation)")
axes[1].set_xlabel("Époque")
axes[1].set_ylim(0, 1)
axes[1].legend()

plt.tight_layout()
plt.savefig("/content/courbes_entrainement.png", dpi=120)
plt.show()


## 8. Évaluer le modèle (rapport + matrice de confusion)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for inputs, targets in val_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_targets.extend(targets.numpy())

print("📊 RAPPORT DE CLASSIFICATION (sur la validation)")
print(classification_report(all_targets, all_preds, target_names=CLASSES, zero_division=0))

cm = confusion_matrix(all_targets, all_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel("Prédit")
plt.ylabel("Réel")
plt.title("Matrice de confusion")
plt.tight_layout()
plt.savefig("/content/matrice_confusion.png", dpi=120)
plt.show()


⚠️ **À lire avant de faire confiance à ce modèle** : avec un déséquilibre de classes comme ici, une bonne précision globale peut cacher une classe minoritaire mal apprise. Regarde bien la ligne de la classe minoritaire dans le rapport ci-dessus (colonne `recall` en particulier) — si elle est proche de 0, le modèle ne sait quasiment pas la reconnaître malgré la pondération, et il faudra annoter plus d'exemples de cette classe avant de faire confiance au modèle en production.

## 9. Sauvegarder le modèle

On sauvegarde à la fois les poids **et** l'ordre des classes (`{f"classes_sexe.json"}`) — indispensable pour réutiliser le modèle plus tard, car `ImageFolder` assigne les indices par ordre alphabétique et ça doit rester cohérent entre entraînement et inférence.

In [ ]:
import json
from google.colab import files

MODEL_NAME = "modele_sexe.pt"
torch.save(model.state_dict(), MODEL_NAME)

with open("classes_sexe.json", "w", encoding="utf-8") as f:
    json.dump(CLASSES, f, ensure_ascii=False, indent=2)

print(f"✅ Modèle sauvegardé : {MODEL_NAME}")
print(f"✅ Classes sauvegardées : classes_sexe.json -> {CLASSES}")

files.download(MODEL_NAME)
files.download(f"classes_sexe.json")


**Prochaine étape** : place `{f"modele_sexe.pt"}` dans `models/sexe/` sur ton PC, et adapte `scripts/utils/pipeline.py` pour charger ce modèle (voir `_load_classifier` / `_classify_crop`, déjà prévus pour ça — il suffit de renseigner `model_sexe_path` lors de l'appel à `CanardIdentificationPipeline`).